# Score quants on a free Google Colab GPU

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saqlain2204/quantgap/blob/main/notebooks/score_quants_colab.ipynb)

This notebook scores GGUF quants against a public quantgap **pack** and saves one result file per quant.
On a free T4 it took about 3 minutes per quant of a 0.5B model (full mode: 160 texts and 48 generations).

Before you run: **Runtime > Change runtime type > T4 GPU**.

Tested on a free Colab T4 runtime (Python 3.13, CUDA 13) on 2026-11-10. Colab images change, so if the install cell fails, the error usually names the missing library.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install

Colab ships CUDA 13, but the prebuilt llama.cpp GPU wheel needs the CUDA 12 runtime. The two `nvidia-*-cu12` packages provide it.

In [ ]:
!pip install -q quantgap
!pip install -q llama-cpp-python --only-binary=:all: --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

## 2. Choose a pack and quants

`PACK` is a quantgap pack (a Hugging Face dataset id or a local folder). Every quant must come from the same model as the pack.

In [ ]:
PACK = "quantgap-packs/Qwen2.5-0.5B-Instruct"
B = "bartowski/Qwen2.5-0.5B-Instruct-GGUF/Qwen2.5-0.5B-Instruct-"
QUANTS = [f"hf://{B}{x}.gguf" for x in ("Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K")]
# Full mode takes about 3 minutes per quant here. Add "--quick" to EXTRA for a 1-minute first look
# (quick results cannot be submitted).
EXTRA = []

## 3. Score

Each quant is a separate `quantgap score` run. Finished results are skipped when you run this cell again, so a dropped session loses nothing.

In [ ]:
import glob
import os
import site
import subprocess
import time

libs = sorted(
    {
        os.path.dirname(p)
        for sp in site.getsitepackages()
        for p in glob.glob(sp + "/nvidia/*/lib/*.so*")
    }
)
env = dict(
    os.environ,
    LD_LIBRARY_PATH=":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")]),
    HF_HUB_DISABLE_SYMLINKS_WARNING="1",
)
os.makedirs("results", exist_ok=True)

for q in QUANTS:
    org = q.removeprefix("hf://").split("/")[0].lower()
    out = f"results/{org}__{os.path.basename(q)[:-5].lower()}.json"
    if os.path.exists(out):
        print("skip (done):", q)
        continue
    t0 = time.time()
    cmd = ["quantgap", "score", q, "--pack", PACK, "--n-gpu-layers", "-1", "--json", out, *EXTRA]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    status = "ok" if r.returncode == 0 else "FAILED"
    print(f"{status:6} {time.time() - t0:5.0f}s  {q}")
    if r.returncode != 0:
        print(r.stdout[-800:], r.stderr[-800:])

## 4. Look at the results

In [ ]:
!quantgap compare results/*.json --chart chart.svg

## 5. Keep them

Download the folder, or send them to the shared results dataset with `quantgap submit` (needs a Hugging Face token with write access, stored as a Colab secret named `HF_TOKEN`):

```
!quantgap submit results/<file>.json --dry-run
```

`--dry-run` shows exactly what would be sent. Colab sessions end, so save your results before closing the tab.

In [ ]:
import shutil

shutil.make_archive("quantgap-results", "zip", "results")
try:
    from google.colab import files

    files.download("quantgap-results.zip")
except ImportError:
    print("saved quantgap-results.zip")